# Part 5 · Metrics

CLIP-T, CLIP-I / DINO-I, DreamSim, LPIPS, layout IoU, sketch recall (+ FID once there are enough images), per mode, from the images of Part 4. The metric models are small, so any GPU (a T4 is fine) or even CPU works.

- **Needs:** `gen_r256.zip` (Part 4) and `pilot_v1.zip` (notebook 01)
- **Produces:** `eval/summary_aligned.csv` + one CSV per mode, in your store
- On a handful of test images these numbers only prove the wiring works. Read them as a table of *directions* (does the sketch raise `sketch_recall`, does the reference raise `dino_i`, does `clip_t` hold up) once you run the full test split.

In [ ]:
# --- bootstrap (identical in every part): find the pipeline files, set paths ---
import os, glob, shutil, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
    assert hits, 'Upload the pipeline/*.py files + pilot_v1.zip as a Kaggle dataset and add it as an input to this notebook.'
    SRC, WORK = os.path.dirname(hits[0]), '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline', '/content'
os.makedirs(f'{WORK}/pipeline', exist_ok=True)
for f in glob.glob(f'{SRC}/*.py'):
    shutil.copy(f, f'{WORK}/pipeline/')
sys.path.insert(0, f'{WORK}/pipeline')
from common import *
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| STORE =', STORE)

In [ ]:
!pip install -q lpips dreamsim "torchmetrics[image]" torch-fidelity

In [ ]:
PILOT = f'{WORK}/pilot_v1'
if not os.path.exists(f'{PILOT}/metadata.jsonl'):
    shutil.unpack_archive(need('pilot_v1.zip'), PILOT)
print(open(f'{PILOT}/manifest.json').read()[:400])

In [ ]:
GEN = f'{WORK}/gen'
shutil.rmtree(GEN, ignore_errors=True)
shutil.unpack_archive(need('gen_r256.zip'), GEN)
print(sorted(os.listdir(GEN)))

In [ ]:
import eval_metrics as EV, pandas as pd
os.makedirs(f'{STORE}/eval', exist_ok=True)
summ = {}
for mode in sorted(os.listdir(GEN)):
    _, s = EV.evaluate_split(PILOT, f'{GEN}/{mode}', 'test', out_csv=f'{STORE}/eval/{mode}.csv')
    summ[mode] = s
table = pd.DataFrame(summ).T
table.to_csv(f'{STORE}/eval/summary_aligned.csv')
table

Columns (higher is better unless noted): `clip_t` caption match · `clip_i`, `dino_i` similarity to the **reference** · `dreamsim`, `lpips` (**lower** is better) distance to the target photo · `layout_iou` object-mask overlap with the target · `sketch_recall` share of sketch strokes that land on an edge of the output, to compare with `sketch_recall_oracle` (the real photo's value, i.e. the ceiling).